In [25]:
import numpy as np 
import pandas as pd 
import os 
import re 
import string 
pd.set_option('future.no_silent_downcasting', True)    #this raises a warning whenever pandas changes your data(such as fillna or changing labels to 0 or 1)

import mlflow 
import setuptools 
import mlflow.sklearn 
import dagshub 
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer 
from sklearn.model_selection import train_test_split 
from sklearn.linear_model import LogisticRegression 
from sklearn.naive_bayes import MultinomialNB
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier 
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from nltk.corpus import stopwords 
import nltk
nltk.download('stopwords')
nltk.download('wordnet')
from nltk.stem import WordNetLemmatizer
import scipy.sparse 

import warnings 
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\PC\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\PC\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [ ]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Get credentials from environment variables
data_path = os.getenv("DATA_PATH", "notebooks/data.csv")
test_size = 0.2 
mlflow_tracking_uri = os.getenv("MLFLOW_TRACKING_URI")
dagshub_repo_owner = os.getenv("DAGSHUB_REPO_OWNER")
dagshub_repo_name = os.getenv("DAGSHUB_REPO_NAME")
experiment_name = "BoW vs Tf-IDF"

In [27]:
mlflow.set_tracking_uri(mlflow_tracking_uri)
dagshub.init(repo_owner = dagshub_repo_owner, repo_name = dagshub_repo_name, mlflow = True)
mlflow.set_experiment(experiment_name)

Accessing as MLayush-dubey

Initialized MLflow to track repo "MLayush-dubey/MLOps-IMDB-Sentiment-Analysis"

Repository MLayush-dubey/MLOps-IMDB-Sentiment-Analysis initialized!

<Experiment: artifact_location='mlflow-artifacts:/22975b197b2343aa8b64592ecab958b9', creation_time=1776149277172, experiment_id='1', last_update_time=1776149277172, lifecycle_stage='active', name='BoW vs Tf-IDF', tags={'mlflow.experimentKind': 'custom_model_development'}>

In [28]:
#Define text preprocessing functions 
def lemmatization(text):
    lemmatizer = WordNetLemmatizer() 
    return " ".join([lemmatizer.lemmatize(word) for word in text.split()])

def remove_stop_words(text):
    stop_words = set(stopwords.words('english'))
    return " ".join([word for word in text.split() if word not in stop_words])

def remove_numbers(text):
    return "".join([char for char in text if not char.isdigit()])

def remove_punctuation(text):
    return re.sub(f"[{re.escape(string.punctuation)}]", '', text)

def lower_case(text):
    return text.lower()

def remove_urls(text):
    return re.sub(r'https?://\S+|www\.\S+', '', text)

def normalize_text(df):
    try:
        df['review'] = df['review'].apply(lower_case)
        df['review'] = df['review'].apply(remove_urls)
        df['review'] = df['review'].apply(remove_punctuation)
        df['review'] = df['review'].apply(remove_numbers)
        df['review'] = df['review'].apply(remove_stop_words)
        df['review'] = df['review'].apply(lemmatization)
        # Remove extra whitespace
        df['review'] = df['review'].apply(lambda x: ' '.join(x.split()))
        return df
    except Exception as e:
        print(f"Error during text normalization: {e}")
        raise

In [29]:
#loading and preprocessing our data
df = pd.read_csv(data_path)
df = normalize_text(df) 
df = df[df['sentiment'].isin(['positive', 'negative'])]
df['sentiment'] = df['sentiment'].replace({'negative': 0, 'positive': 1}).infer_objects(copy = False)
df.head()

,review,sentiment
0,yep lot shouting screaming cheering arguing ce...,0
1,ill honest yall junior high school sitcom firs...,1
2,liked solino much heartrending story italian f...,1
3,seeing forever hollywood would natural want se...,0
4,show great episode one terrible episode hard f...,0


In [30]:
#define vectors and models
VECTORIZERS = {
    'BoW': CountVectorizer(),
    'TF-IDF': TfidfVectorizer()
}

ALGORITHMS = {
    'LogisticRegression': LogisticRegression(),
    'MultinomialNB': MultinomialNB(),
    'XGBoost': XGBClassifier(),
    'RandomForest': RandomForestClassifier(),
    'GradientBoosting': GradientBoostingClassifier()
}

In [32]:
#Setting up MLFLow configs

with mlflow.start_run(run_name = "All experiments") as parent_run:  #All experiment naam ke folder ke andhar 5 exps and 2 vectorizers would run
    for algo_name, algorithm in ALGORITHMS.items():
        for vec_name, vectorizer in VECTORIZERS.items():
            with mlflow.start_run(run_name = f"{algo_name} with {vec_name}", nested = True) as child_run:  #starts a child run inside the parent run for each combination
                try:
                    #feature extraction
                    X = vectorizer.fit_transform(df['review'])
                    y = df['sentiment']
                    X_train, X_test, y_train, y_test= train_test_split(X, y, test_size = 0.2, random_state = 42)

                    #log preprocessing params 
                    mlflow.log_params({
                        "vectorizer": vec_name,
                        "algorithm": algo_name,
                        "test_size": test_size
                    })

                    #Train
                    algorithm.fit(X_train, y_train)

                    #log model specific hyperparameters
                    if algo_name == "LogisticRegression":
                        mlflow.log_param("C", algorithm.C)
                    elif algo_name == "XGBoost":
                        mlflow.log_params({"n_estimators": algorithm.n_estimators, "learning_rate": algorithm.learning_rate})
                    elif algo_name == "MultinomialNB":
                        mlflow.log_param("alpha", algorithm.alpha)
                    elif algo_name == "RandomForest":
                        mlflow.log_params({"n_estimators": algorithm.n_estimators, "max_depth":algorithm.max_depth})
                    elif algo_name == "GradientBoostingClassifier":
                        mlflow.log_params({"n_estimators": algorithm.n_estimators, "learning_rate": algorithm.learning_rate, "max_depth": algorithm.max_depth})

                    
                    #Evaluate 
                    y_pred = algorithm.predict(X_test)
                    metrics = {
                        "accuracy": accuracy_score(y_test, y_pred),
                        "precision": precision_score(y_test, y_pred),
                        "recall": recall_score(y_test, y_pred),
                        "f1": f1_score(y_test, y_pred)
                    }
                    mlflow.log_metrics(metrics)


                    #log model 
                    input_sample = X_test[:5] if not scipy.sparse.issparse(X_test) else X_test[:5].toarray()
                    #take first 5 rows if not sparse else convert to an array then take first 5 rows
                    #Since cv and tfidf both create sparse matrices
                    mlflow.sklearn.log_model(algorithm, "model", input_example = input_sample)

                    print(f"\nAlgorithm: {algo_name}, Vectorizer: {vec_name}")
                    print(f"Metrics: {metrics}")

                except Exception as e:
                    print(f"Error in training {algo_name} with {vec_name}: {e}")
                    mlflow.log_param("error", str(e))


Algorithm: LogisticRegression, Vectorizer: BoW
Metrics: {'accuracy': 0.78, 'precision': 0.76, 'recall': 0.7916666666666666, 'f1': 0.7755102040816326}
🏃 View run LogisticRegression with BoW at: https://dagshub.com/MLayush-dubey/MLOps-IMDB-Sentiment-Analysis.mlflow/#/experiments/1/runs/7c6ae5f462b044be889a5155e4c6e9ae
🧪 View experiment at: https://dagshub.com/MLayush-dubey/MLOps-IMDB-Sentiment-Analysis.mlflow/#/experiments/1



Algorithm: LogisticRegression, Vectorizer: TF-IDF
Metrics: {'accuracy': 0.85, 'precision': 0.9230769230769231, 'recall': 0.75, 'f1': 0.8275862068965517}
🏃 View run LogisticRegression with TF-IDF at: https://dagshub.com/MLayush-dubey/MLOps-IMDB-Sentiment-Analysis.mlflow/#/experiments/1/runs/06cc3f01eaf64529b50e1ca45291bf1b
🧪 View experiment at: https://dagshub.com/MLayush-dubey/MLOps-IMDB-Sentiment-Analysis.mlflow/#/experiments/1



Algorithm: MultinomialNB, Vectorizer: BoW
Metrics: {'accuracy': 0.78, 'precision': 0.825, 'recall': 0.6875, 'f1': 0.75}
🏃 View run MultinomialNB with BoW at: https://dagshub.com/MLayush-dubey/MLOps-IMDB-Sentiment-Analysis.mlflow/#/experiments/1/runs/e791b760dd3f434f8eea0b8e70635a78
🧪 View experiment at: https://dagshub.com/MLayush-dubey/MLOps-IMDB-Sentiment-Analysis.mlflow/#/experiments/1



Algorithm: MultinomialNB, Vectorizer: TF-IDF
Metrics: {'accuracy': 0.77, 'precision': 0.9310344827586207, 'recall': 0.5625, 'f1': 0.7012987012987013}
🏃 View run MultinomialNB with TF-IDF at: https://dagshub.com/MLayush-dubey/MLOps-IMDB-Sentiment-Analysis.mlflow/#/experiments/1/runs/3bdc85332bb74de782038c4421fa4ebe
🧪 View experiment at: https://dagshub.com/MLayush-dubey/MLOps-IMDB-Sentiment-Analysis.mlflow/#/experiments/1



Algorithm: XGBoost, Vectorizer: BoW
Metrics: {'accuracy': 0.73, 'precision': 0.7058823529411765, 'recall': 0.75, 'f1': 0.7272727272727273}
🏃 View run XGBoost with BoW at: https://dagshub.com/MLayush-dubey/MLOps-IMDB-Sentiment-Analysis.mlflow/#/experiments/1/runs/7e016564ebc84996b5981ce2ad550412
🧪 View experiment at: https://dagshub.com/MLayush-dubey/MLOps-IMDB-Sentiment-Analysis.mlflow/#/experiments/1



Algorithm: XGBoost, Vectorizer: TF-IDF
Metrics: {'accuracy': 0.7, 'precision': 0.6875, 'recall': 0.6875, 'f1': 0.6875}
🏃 View run XGBoost with TF-IDF at: https://dagshub.com/MLayush-dubey/MLOps-IMDB-Sentiment-Analysis.mlflow/#/experiments/1/runs/272e3f3305a74e8084e8918540f68f8b
🧪 View experiment at: https://dagshub.com/MLayush-dubey/MLOps-IMDB-Sentiment-Analysis.mlflow/#/experiments/1



Algorithm: RandomForest, Vectorizer: BoW
Metrics: {'accuracy': 0.78, 'precision': 0.8611111111111112, 'recall': 0.6458333333333334, 'f1': 0.7380952380952381}
🏃 View run RandomForest with BoW at: https://dagshub.com/MLayush-dubey/MLOps-IMDB-Sentiment-Analysis.mlflow/#/experiments/1/runs/a44c3a3b0dd7474c9cb44e45610c05aa
🧪 View experiment at: https://dagshub.com/MLayush-dubey/MLOps-IMDB-Sentiment-Analysis.mlflow/#/experiments/1



Algorithm: RandomForest, Vectorizer: TF-IDF
Metrics: {'accuracy': 0.77, 'precision': 0.7906976744186046, 'recall': 0.7083333333333334, 'f1': 0.7472527472527473}
🏃 View run RandomForest with TF-IDF at: https://dagshub.com/MLayush-dubey/MLOps-IMDB-Sentiment-Analysis.mlflow/#/experiments/1/runs/8e765cca1b5440f8b737b6752ac32504
🧪 View experiment at: https://dagshub.com/MLayush-dubey/MLOps-IMDB-Sentiment-Analysis.mlflow/#/experiments/1



Algorithm: GradientBoosting, Vectorizer: BoW
Metrics: {'accuracy': 0.74, 'precision': 0.7037037037037037, 'recall': 0.7916666666666666, 'f1': 0.7450980392156863}
🏃 View run GradientBoosting with BoW at: https://dagshub.com/MLayush-dubey/MLOps-IMDB-Sentiment-Analysis.mlflow/#/experiments/1/runs/05524ffcf2344c2d90cb7690d74ffabb
🧪 View experiment at: https://dagshub.com/MLayush-dubey/MLOps-IMDB-Sentiment-Analysis.mlflow/#/experiments/1



Algorithm: GradientBoosting, Vectorizer: TF-IDF
Metrics: {'accuracy': 0.71, 'precision': 0.7021276595744681, 'recall': 0.6875, 'f1': 0.6947368421052632}
🏃 View run GradientBoosting with TF-IDF at: https://dagshub.com/MLayush-dubey/MLOps-IMDB-Sentiment-Analysis.mlflow/#/experiments/1/runs/2bfde1fc2bfc4788b0372d8ddb9f2a05
🧪 View experiment at: https://dagshub.com/MLayush-dubey/MLOps-IMDB-Sentiment-Analysis.mlflow/#/experiments/1
🏃 View run All experiments at: https://dagshub.com/MLayush-dubey/MLOps-IMDB-Sentiment-Analysis.mlflow/#/experiments/1/runs/43bce4c3da554fa49ac6c900bc8d4ace
🧪 View experiment at: https://dagshub.com/MLayush-dubey/MLOps-IMDB-Sentiment-Analysis.mlflow/#/experiments/1
